# Apache Spark from Groovy (Spark Connect)

Spark's strategic thin-client mode: no driver in the kernel at all — just a
small gRPC client talking to a Spark Connect server. DataFrames, SQL and even
**Groovy-compiled UDFs** (via artifact upload) work; the RDD API and
`SparkContext` do not exist over Connect, by design.

**Server**: download a Spark ≥ 4.2 distribution and run
`sbin/start-connect-server.sh` (listens on `sc://localhost:15002`).

**Kernel flags**: the Connect client deserializes results with Apache Arrow,
which needs JVM access flags; and shipping cell-compiled classes to the server
needs the kernel's class-output directory. Install the kernelspec with:

```
./gradlew installKernelSpec -PjavaHome=/path/to/jdk25 -PjvmArgs='\
  --add-opens=java.base/java.nio=ALL-UNNAMED \
  --sun-misc-unsafe-memory-access=allow \
  -Dio.netty.tryReflectionSetAccessible=true \
  -Dorg.sparkproject.io.netty.tryReflectionSetAccessible=true \
  -Dgroovy.jupyter.classOutputDir=$HOME/.groovy-jupyter/classes'
```

(`--sun-misc-unsafe-memory-access` exists on JDK 24+ only — omit it on 17/21.
Both netty property spellings are included because Spark shades netty.)

In [1]:
// fail fast if the Connect server isn't up (the gRPC client would otherwise
// retry with exponential backoff for minutes)
try {
    new Socket('localhost', 15002).close()
    'Spark Connect server is listening on localhost:15002'
} catch (IOException e) {
    throw new IllegalStateException('''No Spark Connect server on localhost:15002 — start one first:
    JAVA_HOME=<jdk17+> $SPARK_HOME/sbin/start-connect-server.sh''')
}

Spark Connect server is listening on localhost:15002

In [2]:
@Grab('org.apache.spark:spark-connect-client-jvm_2.13:4.2.0')
import org.apache.spark.sql.SparkSession
spark = SparkSession.builder().remote('sc://localhost:15002').getOrCreate()
"Connected to Spark ${spark.version()} — no driver in this JVM"

Connected to Spark 4.2.0 — no driver in this JVM

## DataFrames and SQL over the wire

Identical API to classic mode; execution happens entirely server-side. File
paths are resolved by the *server* (same machine here, so an absolute path to
the whisky CSV works):

In [3]:
whisky = spark.read().option('header', true).option('inferSchema', true)
        .csv(new File('whiskey.csv').absolutePath)
whisky.createOrReplaceTempView('whisky')
"${whisky.count()} rows"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties


86 rows

In [4]:
asRows = { df -> df.collectAsList().collect { row ->
    (0..<row.size()).collectEntries { i -> [df.columns()[i], row.get(i)] } } }
asRows(spark.sql('select Distillery, Smoky, Body from whisky order by Smoky desc, Body desc limit 5'))

Distillery,Smoky,Body
Ardbeg,4,4
Laphroig,4,4
Lagavulin,4,4
Caol Ila,4,3
Talisker,3,4


## Groovy UDFs over Connect

The server knows nothing about classes compiled in kernel cells — a naive UDF
registration fails with
`SparkClassNotFoundException: ... call session.addArtifact`. The recipe:
the kernel (started with `groovy.jupyter.classOutputDir`) writes every
cell-compiled class to disk, so upload the **Groovy runtime jar** (once) and
the **UDF's class file**, then register as usual:

In [5]:
import org.apache.spark.sql.api.java.UDF2
import org.apache.spark.sql.types.DataTypes

class Peatiness implements UDF2<Integer, Integer, Integer>, Serializable {
    Integer call(Integer smoky, Integer medicinal) { smoky * 2 + medicinal }
}

// once per session: the Groovy runtime, so GroovyObject etc. resolve server-side
groovyJar = new File(GroovySystem.class.protectionDomain.codeSource.location.toURI())
spark.addArtifact(groovyJar.absolutePath)

// per UDF class: the kernel-compiled class file
classDir = new File(System.getProperty('groovy.jupyter.classOutputDir'))
spark.addArtifact(new File(classDir, 'Peatiness.class').absolutePath)

spark.udf().register('peatiness', new Peatiness(), DataTypes.IntegerType)
asRows(spark.sql('''
    select Distillery, peatiness(Smoky, Medicinal) as peaty
    from whisky order by peaty desc limit 5
'''))

Distillery,peaty
Ardbeg,12
Lagavulin,12
Laphroig,12
Caol Ila,10
Clynelish,9


The same `addArtifact` calls work against a real remote cluster — that is the
whole point: what leaves the client is exactly what you choose to upload.

See the classic-mode companion notebook (`spark.ipynb`) for the in-kernel
driver variant and the `spark.repl.class.outputDir` cluster recipe. Finally,
close the client session:

In [6]:
spark.stop()
'Connect session closed'

Connect session closed